# Image Segmentation: Global and Adaptive Binarization

This notebook focuses on image segmentation techniques for separating foreground objects from the background (Region of Interest — ROI).

The project traces the evolution of binarization approaches—from simple histogram-based thresholding, through automatic threshold optimization (iterative method and Otsu's method), to advanced local methods (Sauvola) that handle image degradation caused by noise and non-uniform scene illumination.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

if not os.path.exists("coins.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/coins.png --no-check-certificate
if not os.path.exists("rice.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png --no-check-certificate
if not os.path.exists("catalogue.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/catalogue.png --no-check-certificate
if not os.path.exists("bart.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/bart.png --no-check-certificate
if not os.path.exists("figura1.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura1.png --no-check-certificate
if not os.path.exists("figura2.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura2.png --no-check-certificate
if not os.path.exists("figura3.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura3.png --no-check-certificate
if not os.path.exists("figura4.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura4.png --no-check-certificate

coins = cv2.imread("coins.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(coins, cmap="gray")
plt.axis("off")
plt.show()

coins_hist = np.histogram(coins, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(coins_hist[1][:-1], coins_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of coins")
plt.show()

figura1 = cv2.imread("figura1.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura1, cmap="gray")
plt.axis("off")
plt.show()

figura1_hist = np.histogram(figura1, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura1_hist[1][:-1], figura1_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura1")
plt.show()  

figura2 = cv2.imread("figura2.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura2, cmap="gray")
plt.axis("off")
plt.show()

figura2_hist = np.histogram(figura2, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura2_hist[1][:-1], figura2_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura2")
plt.show()

figura3 = cv2.imread("figura3.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura3, cmap="gray")
plt.axis("off")
plt.show()      
figura3_hist = np.histogram(figura3, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura3_hist[1][:-1], figura3_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura3")
plt.show()

figura4 = cv2.imread("figura4.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura4, cmap="gray")
plt.axis("off")
plt.show()
figura4_hist = np.histogram(figura4, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura4_hist[1][:-1], figura4_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura4")
plt.show()  


## 1. Global Binarization (Manual Bimodal Analysis)

Under ideal lighting, the image histogram exhibits two distinct peaks (bimodality)—one for objects and one for the background. Setting the cutoff threshold between these peaks enables fast segmentation.

In [ ]:
def binaryzacja(img, threshold):
    result = np.zeros(img.shape, dtype=np.uint8)
    result[img > threshold] = 255
    plt.imshow(result, cmap="gray")
    plt.axis("off")
    plt.show()
    return result


binary_coins = binaryzacja(coins, 79)

### Noise and Illumination Gradient Effects

The tests below show how environmental disturbances affect a fixed binarization threshold. Gaussian noise or non-uniform illumination (e.g., lens vignetting) blurs the boundary between background and foreground.

In [ ]:
binary_figura1 = binaryzacja(figura1, 79)
binary_figura2 = binaryzacja(figura2, 110)
binary_figura3 = binaryzacja(figura3, 170)
binary_figura4 = binaryzacja(figura4, 46)


In [ ]:
"This is not feasible in all cases: noise can push background pixels above the threshold, so they are classified as objects. If objects have brightness similar to the background, they may disappear after binarization."

## 2. Automatic Threshold Optimization

In industrial settings, manual threshold tuning is impractical. Below are two methods for automatic cutoff estimation:

**1. Iterative Algorithm:** Starts from the mean image intensity and splits pixels into two classes. It then recomputes class means and updates the threshold until convergence.

**2. Otsu's Method (Maximum Variance):** An exhaustive, optimal search over the histogram for threshold $k$ that maximizes between-class variance $\sigma^2_B(k)$, minimizing misclassification at class boundaries.

In [ ]:
def iterative_thresholding(img):
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    
    N = img.shape[0] * img.shape[1]
    p_i = hist / N
    
    P_0 = np.cumsum(p_i)
    i_vals = np.arange(256)
    
    k = int(np.dot(i_vals, p_i))

    while True:
        P0_k = P_0[k]
        P1_k = 1.0 - P0_k

        if P0_k == 0 or P1_k == 0:
            break
        m0 = (1 / P0_k) * np.dot(i_vals[:k+1], p_i[:k+1])

        m1 = (1 / P1_k) * np.dot(i_vals[k+1:], p_i[k+1:])
        
        k_new = int((m0 + m1) / 2)

        if abs(k_new - k) < 1:
            break
            
        k = k_new

    _, binarized = cv2.threshold(img, k, 255, cv2.THRESH_BINARY)

    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Binarization (Estimated threshold: {k})")
    plt.axis('off')
    plt.show()

    return k

images_to_test = [
coins, figura1, figura2, figura3, figura4
]

for img_name in images_to_test:
    iterative_thresholding(img_name)


In [ ]:
def otsu_thresholding(img):
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    
    N = img.shape[0] * img.shape[1]
    p_i = hist / N
    
    i_vals = np.arange(256)
    m_G = np.dot(i_vals, p_i)

    sigma_b_squared = np.zeros(256)
    P_0 = 0.0
    m_k = 0.0

    for k in range(256):
        P_0 += p_i[k]
        m_k += k * p_i[k]
        
        if 0 < P_0 < 1:
            sigma_b_squared[k] = ((m_G * P_0 - m_k) ** 2) / (P_0 * (1.0 - P_0))

    k_bar = int(np.argmax(sigma_b_squared))

    _, binarized = cv2.threshold(img, k_bar, 255, cv2.THRESH_BINARY)
    k_cv2, binarized_cv2 = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    plt.figure(figsize=(16, 4))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.axis('off')
    
    plt.subplot(1, 4, 2)
    plt.plot(sigma_b_squared)
    plt.title("Variance")
    
    plt.subplot(1, 4, 3)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Otsu (custom): {k_bar}")
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(binarized_cv2, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Otsu (CV2): {int(k_cv2)}")
    plt.axis('off')
    plt.show()

    return k_bar

rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)
catalogue = cv2.imread("catalogue.png", cv2.IMREAD_GRAYSCALE)

images_to_test = [
    coins, figura1, figura2, figura3, figura4, rice, catalogue
]

for img_name in images_to_test:
    otsu_thresholding(img_name)

## 3. Illumination Gradient Compensation (Local Binarization)

Global algorithms degrade sharply under directional lighting or vignetting (see `rice.png`). The adaptive (local) approach computes a per-pixel cutoff from statistics in a sliding neighborhood (analysis window).

In [ ]:
def local_binarization(img, W=15):
    X, Y = img.shape
    binarized = np.zeros((X, Y), dtype=np.uint8)
    
    half_W = int(W / 2)

    for i in range(half_W, X - half_W):
        for j in range(half_W, Y - half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]
            threshold = np.mean(window)
            
            if img[i, j] >= threshold:
                binarized[i, j] = 255
            else:
                binarized[i, j] = 0

    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Local (W={W})")
    plt.axis('off')
    plt.show()

images_to_test = [rice, catalogue]

for img in images_to_test:
    local_binarization(img, 15)

### Sauvola-Pietikäinen Algorithm

A plain moving average often fails on uniform regions (e.g., clean text backgrounds), producing false positives. Sauvola incorporates local standard deviation: in low-variance areas the threshold is lowered, flattening the background while preserving sharp text edges.

In [ ]:
def sauvola_binarization(img, W=15, k=0.15, R=128, sign=1):
    X, Y = img.shape
    binarized = np.zeros((X, Y), dtype=np.uint8)
    
    half_W = int(W / 2)

    for i in range(half_W, X - half_W):
        for j in range(half_W, Y - half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]
            m = np.mean(window)
            s = np.std(window)
            
            threshold = m * (1 + sign * k * (s / R - 1))
            
            if img[i, j] >= threshold:
                binarized[i, j] = 255
            else:
                binarized[i, j] = 0

    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    sign_str = "+" if sign == 1 else "-"
    plt.title(f"Sauvola (W={W}, k={k}, {sign_str})")
    plt.axis('off')
    plt.show()

sauvola_binarization(rice, W=15, k=0.15, R=128, sign=-1)
sauvola_binarization(rice, W=15, k=0.15, R=128, sign=1)
sauvola_binarization(catalogue, W=15, k=0.15, R=128, sign=1)
sauvola_binarization(catalogue, W=15, k=0.15, R=128, sign=-1)

## 4. Range-Based Segmentation (Double Thresholding)

Unlike single-threshold above/below cutoff, double thresholding defines a brightness window. It selectively isolates pixels in a specific intensity band and, combined with AND logic, is useful for tasks such as skin-color detection.

In [ ]:
bart = cv2.imread('bart.png', cv2.IMREAD_GRAYSCALE)

hist = cv2.calcHist([bart], [0], None, [256], [0, 256])

plt.subplot(1, 2, 1)
plt.imshow(bart, cmap='gray', vmin=0, vmax=255)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.plot(hist)
plt.title("Histogram")
plt.show()

LOWER = 185 
UPPER = 216

mask_lower = (bart >= LOWER).astype(int)
mask_upper = (bart <= UPPER).astype(int)

binarized = (mask_lower * mask_upper * 255).astype(np.uint8)

plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
plt.title(f"Segmentation ({LOWER} - {UPPER})")
plt.axis('off')
plt.show()

### Cleaning Temporary Files

Removes configuration files and images downloaded for local notebook testing.

In [ ]:
import os
import glob

trash_files = glob.glob('*.png*') + glob.glob('*.jpg*') + glob.glob('*.bmp*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Cleanup complete! Workspace cleared of temporary files.")